# Практика 2 — качество данных

## Подготовка

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

## Два входных файла

In [ ]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

## Профиль входа

In [ ]:
print(export.isna().sum())
print(export["action_taken"].value_counts(dropna=False))

## A. Проверки входа

In [ ]:
print(export["source_row"].duplicated().sum())
print(export.loc[~export["action_taken"].isin(["1", "2", "3"]), ["source_row", "action_taken"]])
print(export["county_code"].value_counts(dropna=False))

## Преобразование суммы

In [ ]:
work = export.copy(deep=True)
work["county_code"] = work["county_code"].str.strip()
for field in ["source_row", "loan_amount", "action_taken"]:
    numbers = pd.to_numeric(work[field], errors="coerce")
    failed = work[field].notna() & numbers.isna()
    print(field, int(failed.sum()))
    print(work.loc[failed, ["source_row", field]])
    work[field] = numbers

## B. Ошибки преобразования дохода

In [ ]:
income_numbers = pd.to_numeric(export["income"], errors="coerce")
income_parse_error = export["income"].notna() & income_numbers.isna()
print(export.loc[income_parse_error, ["source_row", "income"]])
print("Новые ошибки:", income_parse_error.sum())
print("Исходные пропуски:", export["income"].isna().sum())
print("После преобразования:", income_numbers.isna().sum())
assert_frame_equal(export, raw_copy)

## Проверка повторной доставки

In [ ]:
known_key = export["source_row"].notna()
repeated = export.loc[known_key].drop_duplicates()
assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
replay_mask = known_key & export["source_row"].duplicated()
unique = work.loc[~replay_mask].copy()
print(len(export), len(unique), int(replay_mask.sum()))

## C. Строки с нарушениями обязательных правил

In [ ]:
bad_core = unique["loan_amount"].isna() | unique["loan_amount"].le(0)
bad_core = bad_core | ~unique["action_taken"].isin([1, 2, 3])
quarantine_part = unique.loc[bad_core].copy()
clean_part = unique.loc[~bad_core].copy()
print(quarantine_part[["source_row", "loan_amount", "action_taken"]])
assert len(export) == len(clean_part) + len(quarantine_part) + replay_mask.sum()
print(len(clean_part), len(quarantine_part))

## Функция по разобранным шагам

In [ ]:
def clean_export(source):
    out = source.copy(deep=True)
    known_key = out["source_row"].notna()
    repeated = out.loc[known_key].drop_duplicates()
    assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
    replay_mask = known_key & out["source_row"].duplicated()
    replay_n = int(replay_mask.sum())
    out = out.loc[~replay_mask].copy()
    out["county_code"] = out["county_code"].str.strip()
    parse_counts = {}
    for field in ["source_row", "loan_amount", "income", "action_taken"]:
        numbers = pd.to_numeric(out[field], errors="coerce")
        parse_counts[field] = int((out[field].notna() & numbers.isna()).sum())
        out[field] = numbers
    reason = pd.Series("", index=out.index, dtype="string")
    reason.loc[out["source_row"].isna()] += "unknown_source_row;"
    bad_amount = out["loan_amount"].isna() | out["loan_amount"].le(0)
    reason.loc[bad_amount] += "invalid_loan_amount;"
    reason.loc[~out["action_taken"].isin([1, 2, 3])] += "invalid_action_taken;"
    bad_county = ~out["county_code"].eq("11001").fillna(False)
    reason.loc[bad_county] += "invalid_county;"
    invalid = reason.ne("")
    clean = out.loc[~invalid].copy()
    quarantine = source.loc[out.index[invalid]].copy()
    quarantine["reason"] = reason.loc[invalid]
    audit = {
        "input": len(source), "replays": replay_n,
        "clean": len(clean), "quarantine": len(quarantine),
        "parse_errors": parse_counts,
    }
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
    return clean, quarantine, audit

## Результат функции

In [ ]:
result, rejected, audit = clean_export(export)
print(audit)
rejected

## D. Сохранность и повторяемость

In [ ]:
again, rejected_again, audit_again = clean_export(export)
assert_frame_equal(result, again)
assert_frame_equal(rejected, rejected_again)
assert audit == audit_again
assert_frame_equal(export, raw_copy)
assert result["source_row"].is_unique
assert result["action_taken"].isin([1, 2, 3]).all()
assert result["loan_amount"].gt(0).all()
assert len(export) == len(result) + len(rejected) + audit["replays"]
print(result["income"].isna().sum())
assert rejected["reason"].ne("").all()

## Показатель и выбранная группа

In [ ]:
def show_approval(label, table):
    n = len(table)
    assert n > 0
    k = table["action_taken"].isin([1, 2]).sum()
    print(label, k, n, f"{100 * k / n:.2f}%")

show_approval("Оригинал", decisions)
show_approval("После очистки", result)
complete = result.dropna(subset=["income"])
show_approval("Известный доход", complete)

## E. Сохранение и отчёт

In [ ]:
import json
result.to_csv(WORK / "clean.csv", index=False)
rejected.to_csv(WORK / "quarantine.csv", index=False)
(WORK / "quality_report.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(sorted(p.name for p in WORK.glob("*")))